<a href="https://colab.research.google.com/github/Shriyamaricharla/Data/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install required libraries
!pip install pandas scikit-learn imbalanced-learn matplotlib interpret requests

# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import numpy as np
from interpret.glassbox import (LogisticRegression, ExplainableBoostingClassifier, ClassificationTree)
from interpret.blackbox import LimeTabular
from sklearn.ensemble import RandomForestClassifier

from interpret import show
from sklearn.metrics import f1_score, accuracy_score
import requests
import pickle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 68.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 142.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.9 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl siz

In [4]:
# URL of the CSV file in the GitHub repository
#file_url = "https://raw.githubusercontent.com/marichala/ML/refs/heads/ExplainableAI/healthcare-dataset-stroke-data.csv"
file_url="https://raw.githubusercontent.com/Shriyamaricharla/Data/main/healthcare-dataset-stroke-data.csv"
# Load the CSV file into a pandas DataFrame
data = pd.read_csv(file_url)
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [5]:
categorical_cols = ["gender",
                            "ever_married",
                            "work_type",
                            "Residence_type",
                            "smoking_status"]

encoded = pd.get_dummies(data[categorical_cols], prefix=categorical_cols)
#encoded.info()

In [6]:
data = pd.concat([encoded, data], axis=1)
data.drop(categorical_cols, axis=1, inplace=True)

#data.info()#

# Impute missing values of BMI
data.bmi = data.bmi.fillna(0)

# Drop id as it is not relevant
if 'id' in data.columns:
  data.drop(['id'], axis=1, inplace=True)

#spliting features (X) and lables (y)
#Features = all columns in the dataset except the last column.... represented :(start default 0) to :-1 (minus 1 column from end)
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

# Split the data for evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=2021)

print("Before oversampling:", X_train.shape)
oversample = RandomOverSampler(sampling_strategy='minority')

# Convert to numpy and oversample
x_np = X_train.to_numpy()
y_np = y_train.to_numpy()

# Convert y to int before oversampling
y_np = y_np.astype(int)

x_np, y_np = oversample.fit_resample(x_np, y_np)

# Convert boolean values to 1.0 or 0.0
x_np = np.where(x_np == True, 1, x_np)
x_np = np.where(x_np == False, 0, x_np)

y_np = np.where(y_np == True, 1, y_np)
y_np = np.where(y_np == False, 0, y_np)

# Convert back to pandas
X_train = pd.DataFrame(x_np, columns=X_train.columns)
y_train = pd.Series(y_np, name=y_train.name)

# Check for non-numeric values in each column
for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        print(f"Train Column '{col}' contains non-numeric values.")
        # Handle non-numeric values (e.g., convert to numeric or remove rows)
        # Example: Convert to numeric, coercing errors to NaN
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        # Or remove rows with non-numeric values:
        # X_train = X_train[pd.to_numeric(X_train[col], errors='coerce').notnull()]

# Convert columns to float if all values are numeric
for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype(float)

# Check for non-numeric values in each column
for col in X_test.columns:
    if not pd.api.types.is_numeric_dtype(X_test[col]):
        print(f"Test Column '{col}' contains non-numeric values.")
        # Handle non-numeric values (e.g., convert to numeric or remove rows)
        # Example: Convert to numeric, coercing errors to NaN
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
        # Or remove rows with non-numeric values:
        # X_train = X_train[pd.to_numeric(X_train[col], errors='coerce').notnull()]

# Convert columns to float if all values are numeric
for col in X_test.columns:
    if pd.api.types.is_numeric_dtype(X_test[col]):
        X_test[col] = X_test[col].astype(float)

X_test.info()
X_train.info()


Before oversampling: (4088, 21)
Train Column 'gender_Female' contains non-numeric values.
Train Column 'gender_Male' contains non-numeric values.
Train Column 'gender_Other' contains non-numeric values.
Train Column 'ever_married_No' contains non-numeric values.
Train Column 'ever_married_Yes' contains non-numeric values.
Train Column 'work_type_Govt_job' contains non-numeric values.
Train Column 'work_type_Never_worked' contains non-numeric values.
Train Column 'work_type_Private' contains non-numeric values.
Train Column 'work_type_Self-employed' contains non-numeric values.
Train Column 'work_type_children' contains non-numeric values.
Train Column 'Residence_type_Rural' contains non-numeric values.
Train Column 'Residence_type_Urban' contains non-numeric values.
Train Column 'smoking_status_Unknown' contains non-numeric values.
Train Column 'smoking_status_formerly smoked' contains non-numeric values.
Train Column 'smoking_status_never smoked' contains non-numeric values.
Train Col

In [8]:
lr = LogisticRegression(random_state=2021, feature_names=X_train.columns, penalty='l1', solver='liblinear')
lr.fit(X_train, y_train)
print("Training finished.")

# %% Evaluate logistic regression model
y_pred = lr.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

# %% Explain local prediction
lr_local = lr.explain_local(X_test[:100], y_test[:100], name='Logistic Regression')
show(lr_local)

Training finished.
F1 Score 0.5162362558186249
Accuracy 0.7348336594911937


In [7]:
ebm = ExplainableBoostingClassifier(random_state=2021)
ebm.fit(X_train, y_train)
print("Training finished.")
y_pred = ebm.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")



KeyboardInterrupt: 

In [1]:
import pandas as pd

# Train the Logistic Regression model (moved from cell qqSAqVymmAS7 to ensure 'lr' is defined)
lr = LogisticRegression(random_state=2021, feature_names=X_train.columns, penalty='l1', solver='liblinear')
lr.fit(X_train, y_train)
print("Logistic Regression Model Training finished.")

# Get feature names from the training data
feature_names = X_train.columns.tolist()

# Create a dictionary to store new record data
new_record_data = {}

print("\nPlease enter values for the following features")
print("You are expected to input either '0' for no and '1' for yes in every column except age, BMI and average glucose level.")
print("You will need to provide the actual numerical data for those sections:")
for feature in feature_names:
    try:
        # Attempt to convert to float directly for numeric inputs
        value = float(input(f"Enter value for '{feature}': "))
        new_record_data[feature] = value
    except ValueError:
        # If not a float, treat as string (e.g., for 'Other' gender that might be '0' or '1' as string initially)
        # Note: In a real application, you'd want more robust type handling and validation
        value = input(f"Enter value for '{feature}' (e.g., 0 or 1 for one-hot encoded features): ")
        try:
            new_record_data[feature] = float(value)
        except ValueError:
            new_record_data[feature] = value # Fallback to string if conversion fails

# Convert the dictionary to a DataFrame, ensuring column order matches X_train
new_record_df = pd.DataFrame([new_record_data])

# Ensure all columns are float type, as per X_train/X_test preprocessing
for col in new_record_df.columns:
    if pd.api.types.is_numeric_dtype(new_record_df[col]):
        new_record_df[col] = new_record_df[col].astype(float)

print("\nNew record created:")
display(new_record_df)

# Predict the likelihood using the Logistic Regression model
# The model expects probabilities for binary classification (likelihood of being class 1)
predicted_likelihood = lr.predict_proba(new_record_df)[:, 1]

print(f"\nPredicted likelihood of stroke for the new record: {predicted_likelihood[0]*100:.2f}%")

# You can also get the class prediction (0 or 1)
predicted_class = lr.predict(new_record_df)
print(f"Predicted class (0=No Stroke, 1=Stroke): {predicted_class[0]}")

lr_local_new_record = lr.explain_local(new_record_df, predicted_class, name='Logistic Regression Local Explanation')
show(lr_local_new_record)

# Access the explanation data for the first instance
explanation_data = lr_local_new_record.data(0)
feature_contributions = dict(zip(explanation_data['names'], explanation_data['scores']))

# Find the feature with the highest absolute contribution
most_influential_feature = None
max_abs_contribution = 0

for feature, contribution in feature_contributions.items():
    if abs(contribution) > max_abs_contribution:
        max_abs_contribution = abs(contribution)
        most_influential_feature = feature

print(f"\nThe most influential factor for this prediction was: '{most_influential_feature}' with a contribution of {max_abs_contribution:.4f}")

print("\nAdvice: To ensure that the risk of getting a stroke is reduced, please ensure that: ")
print(" - you are exercising 20-30 minutes per day")
print(" - your BMI is between 18.5 and 27.5")
print(" - alcohol consumption is reduced")
print(" - blood pressure is regulated")
print(" - smoking habits are abandoned")
print("If you are concerned about your risk of having a stroke, please contact your local GP")


NameError: name 'LogisticRegression' is not defined